In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load the dataset
try:
    kalimati_df = pd.read_csv("kalimati_historical_dataset.csv")
    print(f"Dataset loaded successfully with {len(kalimati_df)} rows")
except FileNotFoundError:
    print("ERROR: kalimati_historical_dataset.csv not found. Please ensure the file is in the same directory.")
    kalimati_df = None

if kalimati_df is not None:
    kalimati_df

In [ ]:
# Standardize column names
kalimati_df.columns = ["Vegetables", "Date", "Weight", "Minimum", "Maximum", "Average"]
kalimati_df

In [ ]:
# Check data info
kalimati_df.info()

In [ ]:
# Convert Date column to datetime
kalimati_df["Date"] = pd.to_datetime(kalimati_df["Date"], yearfirst=True, format='mixed')
print("Date conversion completed")

In [ ]:
kalimati_df

In [ ]:
# Convert Weight to lowercase
kalimati_df["Weight"] = kalimati_df["Weight"].str.lower()
kalimati_df

In [ ]:
# Clean price columns - remove 'Rs' and convert to numeric
for col in ["Minimum", "Maximum", "Average"]:
    kalimati_df[col] = kalimati_df[col].astype(str).str.replace("Rs", "").str.strip()
    kalimati_df[col] = pd.to_numeric(kalimati_df[col], errors='coerce')
    
print("Price columns cleaned")

In [ ]:
# Remove invalid records with zero or null average prices
kalimati_df = kalimati_df[kalimati_df["Average"] > 0].dropna(subset=["Average"])
print(f"Dataset cleaned: {len(kalimati_df)} valid records remaining")
kalimati_df

In [ ]:
kalimati_df.info()

In [ ]:
# Check unique vegetables
num_vegetables = kalimati_df["Vegetables"].nunique()
print(f"Total unique vegetables: {num_vegetables}")
print("\nVegetables list:")
print(kalimati_df["Vegetables"].unique())

In [ ]:
# Get most traded vegetables
highest_traded_vegetables = kalimati_df["Vegetables"].value_counts()
print("Top 10 most traded vegetables:")
print(highest_traded_vegetables.head(10))

In [ ]:
# Visualize top 5 traded vegetables price trends
top_traded_vegetables = highest_traded_vegetables.head(5).index.to_list()
fig, axes = plt.subplots(5, 1, figsize=(14, 18))

for i, vegies in enumerate(top_traded_vegetables):
    subset = kalimati_df[kalimati_df["Vegetables"] == vegies]
    axes[i].plot(subset["Date"], subset["Average"], color='green', linewidth=2, label='Average')
    axes[i].fill_between(subset["Date"], subset["Minimum"], subset["Maximum"], alpha=0.2, color="red", label='Min-Max Range')
    axes[i].set_title(f"{vegies} - Price Trend", fontsize=12, fontweight='bold')
    axes[i].set_ylabel("NPR/KG", fontsize=10)
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('price_trends.png', dpi=300, bbox_inches='tight')
plt.show()
print("Price trends visualization saved as 'price_trends.png'")

In [ ]:
# Sample: Filter data for one vegetable
d = kalimati_df[kalimati_df["Vegetables"] == "Cauli Local"].copy()
d = d.sort_values("Date").reset_index(drop=True)
print(f"Sample data for 'Cauli Local': {len(d)} records")
d.head()

In [ ]:
# Feature Engineering Function
def make_features(kalimati_df, Vegetables):
    """Create lag, rolling, temporal, and seasonal features for price prediction"""
    d = kalimati_df[kalimati_df["Vegetables"] == Vegetables].copy()
    d = d.sort_values("Date").reset_index(drop=True)
    
    # Lag features
    d["lag_1"] = d["Average"].shift(1)
    d["lag_7"] = d["Average"].shift(7)
    d["lag_30"] = d["Average"].shift(30)

    # Rolling statistics
    d["roll_mean_7"] = d["Average"].rolling(7).mean()
    d["roll_std_7"] = d["Average"].rolling(7).std()
    d["roll_mean_30"] = d["Average"].rolling(30).mean()
    d["roll_std_30"] = d["Average"].rolling(30).std()

    # Temporal features
    d["month"] = d["Date"].dt.month
    d["day_of_week"] = d["Date"].dt.dayofweek
    d["day_of_year"] = d["Date"].dt.dayofyear

    # Seasonal indicators
    d["is_dashain"] = d["month"].isin([9, 10]).astype(int)
    d["is_monsoon"] = d["month"].isin([6, 7, 8]).astype(int)

    # Price momentum
    d["pct_change"] = d["Average"].pct_change(7)
    
    # Remove rows with NaN values
    d = d.dropna()
    return d

print("Feature engineering function defined")

In [ ]:
# Create features for all vegetables
all_features = []
failed_vegetables = []

for vegetable in kalimati_df["Vegetables"].unique():
    try:
        features_df = make_features(kalimati_df, vegetable)
        if len(features_df) > 30:
            all_features.append(features_df)
    except Exception as e:
        failed_vegetables.append((vegetable, str(e)))

master = pd.concat(all_features, ignore_index=True)
print(f"Features created for {len(all_features)} vegetables")
print(f"Failed vegetables: {len(failed_vegetables)}")
print(f"Total records: {len(master)}")

In [ ]:
master.head()

In [ ]:
# Anomaly Detection using IQR method
def label_spikes(d):
    """Label price anomalies using rolling IQR method"""
    q1 = d["Average"].rolling(window=30, min_periods=1).quantile(0.25)
    q3 = d["Average"].rolling(window=30, min_periods=1).quantile(0.75)
    iqr = q3 - q1

    uthreshold = q3 + 1.5 * iqr
    lthreshold = q1 - 1.5 * iqr

    d["anomaly_label"] = 0
    d.loc[d["Average"] > uthreshold, "anomaly_label"] = 1
    d.loc[d["Average"] < lthreshold, "anomaly_label"] = -1

    return d

master = master.reset_index(drop=True)
master = master.groupby("Vegetables", group_keys=False).apply(label_spikes)
print("Anomaly detection completed")

In [ ]:
print("Anomaly label distribution:")
print(master["anomaly_label"].value_counts())
print(f"\nTotal records: {len(master)}")
master.head()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

# Define features
FEATURES = [
    "lag_1", "lag_7", "lag_30",
    "roll_mean_7", "roll_std_7", "roll_mean_30", "roll_std_30",
    "month", "day_of_week", "day_of_year",
    "is_dashain", "is_monsoon", "pct_change",
    "anomaly_label"
]

le = LabelEncoder()
master["Vegetables_enc"] = le.fit_transform(master["Vegetables"])
FEATURES_FULL = FEATURES + ["Vegetables_enc"]

X = master[FEATURES_FULL]
y = master["Average"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
# Train model
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

print("Training model...")
model.fit(X_train, y_train)
print("Model training completed!")

# Predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Evaluate
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)
train_mape = mean_absolute_percentage_error(y_train, y_pred_train)
test_mape = mean_absolute_percentage_error(y_test, y_pred_test)
train_mae = mean_absolute_error(y_train, y_pred_train)
test_mae = mean_absolute_error(y_test, y_pred_test)

print("\n" + "="*50)
print("MODEL PERFORMANCE")
print("="*50)
print(f"Train R²: {train_r2:.4f} | Test R²: {test_r2:.4f}")
print(f"Train MAPE: {train_mape:.4f} | Test MAPE: {test_mape:.4f}")
print(f"Train MAE: {train_mae:.2f} NPR/KG | Test MAE: {test_mae:.2f} NPR/KG")
print("="*50)

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': FEATURES_FULL,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Important Features:")
print(feature_importance.head(10))

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'].head(10), feature_importance['importance'].head(10), color='skyblue')
plt.xlabel('Importance Score')
plt.title('Top 10 Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot saved as 'feature_importance.png'")

In [ ]:
# Save models
models_dir = os.path.join(os.getcwd(), 'models')
os.makedirs(models_dir, exist_ok=True)

model_path = os.path.join(models_dir, 'random_forest_model.pkl')
encoder_path = os.path.join(models_dir, 'label_encoder.pkl')

try:
    joblib.dump(model, model_path)
    joblib.dump(le, encoder_path)
    print(f"Model saved to: {model_path}")
    print(f"Encoder saved to: {encoder_path}")
except Exception as e:
    print(f"Error saving models: {e}")

In [ ]:
d.sort_values("Date").iloc[-1]

In [ ]:
def forecast_7days_direct(Vegetables_name, kalimati_df, model, le):
    """Forecast vegetable prices for 7 days"""
    try:
        d = make_features(kalimati_df, Vegetables_name)
        d = label_spikes(d)
        last_row = d.iloc[-1].copy()
        last_row['Vegetables_enc'] = le.transform([Vegetables_name])[0]
        
        MODEL_FEATURES = [
            'lag_1', 'lag_7', 'lag_30',
            'roll_mean_7', 'roll_std_7', 'roll_mean_30', 'roll_std_30',
            'month', 'day_of_week', 'day_of_year',
            'is_dashain', 'is_monsoon', 'pct_change',
            'anomaly_label', 'Vegetables_enc'
        ]
        
        forecasts = []
        for day in range(1, 8):
            row = pd.DataFrame([last_row[MODEL_FEATURES]])
            pred = model.predict(row)[0]
            forecasts.append(round(float(pred), 2))
            
            last_row['lag_30'] = last_row['lag_7']
            last_row['lag_7'] = last_row['lag_1']
            last_row['lag_1'] = pred
        
        return forecasts
    
    except Exception as e:
        print(f"Error forecasting for {Vegetables_name}: {e}")
        return None

print("Forecasting function defined")

In [ ]:
# Example forecast
vegetable_to_forecast = 'Tomato Big(Nepali)'

if vegetable_to_forecast in kalimati_df['Vegetables'].unique():
    preds = forecast_7days_direct(vegetable_to_forecast, kalimati_df, model, le)
    
    if preds:
        print(f"\n{'='*50}")
        print(f"7-DAY FORECAST: {vegetable_to_forecast}")
        print(f"{'='*50}")
        for day, price in enumerate(preds, 1):
            print(f"Day {day}: {price} NPR/KG")
        print(f"{'='*50}")
else:
    print(f"Vegetable '{vegetable_to_forecast}' not found.")

In [ ]:
# Forecasts for all vegetables
all_forecasts = {}

print("Generating forecasts...\n")
for vegetable in kalimati_df['Vegetables'].unique()[:10]:
    forecast = forecast_7days_direct(vegetable, kalimati_df, model, le)
    if forecast:
        all_forecasts[vegetable] = forecast
        print(f"✓ {vegetable}: {forecast}")

print(f"\nForecasts generated for {len(all_forecasts)} vegetables")

In [ ]:
print("\n" + "="*60)
print("PROJECT SUMMARY")
print("="*60)
print(f"Total records: {len(master)}")
print(f"Vegetables: {master['Vegetables'].nunique()}")
print(f"Model: Random Forest (300 trees)")
print(f"Test MAPE: {test_mape:.4f}")
print(f"Test R²: {test_r2:.4f}")
print(f"\nFiles saved:")
print(f"  - Model: {model_path}")
print(f"  - Encoder: {encoder_path}")
print(f"  - Feature importance plot: feature_importance.png")
print(f"  - Price trends plot: price_trends.png")
print("="*60)